# Class 3 - Building a Simple Agent

**Week 5: Introduction to AI Agents**

### Learning objectives
By the end of this notebook you will be able to:
- Explain why short-term memory matters across turns.
- Build a multi-tool LangChain agent on Groq (calculator + mock search).
- Add guardrails: recursion limits and safe tool error strings.
- Know when to look at **LangGraph** (complex graphs) or **LlamaIndex** (another agent stack) after this course pattern.

Week 6 will cover retrieval / vector memory — keep memory here to chat history + optional scratch notes.

## Setup

In [ ]:
!pip install -q langchain langchain-groq langgraph

In [ ]:
import os
import re
import uuid

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print("No GROQ_API_KEY found. Live agent cells will skip.")
else:
    print("Found GROQ_API_KEY. Ready to build the agent.")

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver


def make_llm():
    if not GROQ_API_KEY:
        return None
    return ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)


def build_memory_agent(tools, system_prompt=None, max_turns=8):
    llm = make_llm()
    if llm is None:
        return None
    prompt = system_prompt or (
        "You are a helpful multi-tool assistant. "
        "Use calculator for math and mock_search for facts in the catalog. "
        "If a tool errors, explain the problem and ask for a corrected input."
    )
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=prompt,
        checkpointer=InMemorySaver(),
    )
    agent._max_turns = max_turns
    return agent


def chat(agent, text: str, thread_id: str = "week5-class3"):
    if agent is None:
        print("Skipping — no agent / key.")
        return None
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={
            "configurable": {"thread_id": thread_id},
            "recursion_limit": getattr(agent, "_max_turns", 8),
        },
    )
    final = result["messages"][-1]
    content = getattr(final, "content", final)
    print(content)
    return result

## 1. Why memory matters

Run the cell below twice in spirit:
1. **Without memory** — remember Kathmandu on one `thread_id`, then ask the follow-up on a **new** `thread_id` (no shared checkpointer history).
2. **With memory** — run both turns on the **same** `thread_id` so `InMemorySaver` keeps the earlier message.

Same follow-up question; only the thread changes. If `GROQ_API_KEY` is missing, the cell prints a skip message.


In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * 1.8 + 5'."""
    expr = expression.strip()
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
        return "Calculator error: only digits and + - * / ( ) are allowed."
    try:
        value = eval(expr, {"__builtins__": {}}, {})
    except Exception as e:
        return f"Calculator error: {e}"
    return str(value)


@tool
def mock_search(query: str) -> str:
    """Look up a short fact from a tiny local catalog (not the real web)."""
    catalog = {
        "kathmandu elevation": "Kathmandu sits about 1,400 meters above sea level.",
        "nepal capital": "Kathmandu is the capital of Nepal.",
        "water boil celsius": "Pure water boils at 100°C at standard pressure.",
    }
    q = query.strip().lower()
    for key, value in catalog.items():
        if key in q or q in key:
            return value
    return "No catalog hit. Try queries like 'Nepal capital' or 'Kathmandu elevation'."


demo_tools = [calculator, mock_search]
agent = build_memory_agent(demo_tools)

remember_msg = "Remember that my destination city is Kathmandu."
follow_up = "Using mock_search, what is the elevation there?"

if agent is None:
    print("Skipping memory contrast — no agent / GROQ_API_KEY.")
else:
    print("=== Without shared history (new thread_id on the follow-up) ===")
    chat(agent, remember_msg, thread_id=f"no-memory-{uuid.uuid4()}")
    chat(agent, follow_up, thread_id=f"no-memory-{uuid.uuid4()}")

    print("\n=== With memory (same thread_id for both turns) ===")
    shared_thread = f"memory-demo-{uuid.uuid4()}"
    chat(agent, remember_msg, thread_id=shared_thread)
    chat(agent, follow_up, thread_id=shared_thread)


## 2. Two kinds of memory (foundations level)

- **Short-term:** the message list for a `thread_id` (what `InMemorySaver` keeps for this process).
- **Scratch notes:** a string or dict *you* maintain and inject into the next user message when needed.

Vector databases / RAG document memory arrive in Week 6 — do not add them here.

In [ ]:
NOTES = {"destination": None}

def note_aware_chat(agent, text: str, thread_id: str = "notes-demo"):
    # Lightweight scratchpad: prepend known notes so the model can use them
    preface = ""
    if NOTES.get("destination"):
        preface = f"(Instructor notes: destination={NOTES['destination']})\n"
    return chat(agent, preface + text, thread_id=thread_id)

NOTES["destination"] = "Pokhara"
note_aware_chat(agent, "Should I search for the capital or stay focused on my destination?")

## 3. Guardrails

- Cap iterations with `recursion_limit` / `max_turns`.
- Return tool errors as strings (calculator already does).
- Keep the tool list explicit — only register tools you intend to allow.

In [ ]:
strict_agent = build_memory_agent(demo_tools, max_turns=5)
chat(strict_agent, "What is 17 * 19? Then remind me of Nepal's capital.", thread_id="guard-demo")

## Closing

You shipped a small agent: tools, memory, and limits. For branching multi-actor workflows, explore **LangGraph** next. If your team standardizes on **LlamaIndex**, the same ideas transfer — tools, a loop, memory, and stop conditions.

**Next week:** Foundations of RAG & Chatbots.

## Challenges

### Challenge 01 — `unit_convert`
Add `@tool def unit_convert(value: float, from_unit: str, to_unit: str) -> str` supporting at least `celsius↔fahrenheit` and `km↔miles`. Register it and test.

In [ ]:
# TODO
pass

### Challenge 02 — Expand `mock_search`
Add three new catalog entries and demonstrate a successful lookup for one of them.

In [ ]:
# TODO
pass

### Challenge 03 — Harden the calculator
Extend validation (e.g. reject `**`, `//`, or empty input) and show the agent recovering from a bad expression via the error string.

In [ ]:
# TODO
pass

### Challenge 04 (optional) — Persistent notes
Update `NOTES` from user text (e.g. parse "my destination is X") and show a later answer that depends on the note.

In [ ]:
# TODO
pass